# Track D — Human Calibration Study

**Goal**: Validate that `QualityMetrics(mode="heuristic")` agrees with an LLM judge (GPT-4o proxy) and ultimately with real human raters.

## Two phases

**Phase 1 (immediate)**: LLM-as-proxy annotator
- 50 contexts sampled across 5 quality tiers
- Score each with heuristic mode AND llm mode
- Compute Spearman ρ between them, per dimension and overall
- Target: ρ > 0.65 = validated; flag dimensions with ρ < 0.4 for improvement

**Phase 2 (deferred — run when annotators available)**
- Load human ratings from `human_ratings.csv` (template provided)
- Compute inter-annotator agreement (Krippendorff's α)
- Correlate human scores vs heuristic scores
- Identify dimensions needing weight adjustment

## Setup
```
pip install python-dotenv litellm scipy pandas matplotlib
```
Set your `.env` file with `OPENAI_API_KEY=sk-...`

In [ ]:
import os, sys, json, pathlib, random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120
from scipy import stats
from dotenv import load_dotenv

load_dotenv(pathlib.Path(__file__).parent.parent.parent / ".env" if "__file__" in dir() else pathlib.Path("../../.env"))
# Add mycontext to path
sys.path.insert(0, str(pathlib.Path("../../src")))

from mycontext import Context
from mycontext.foundation import Guidance, Directive, Constraints
from mycontext.intelligence import QualityMetrics, QualityDimension

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

heuristic_metrics = QualityMetrics(mode="heuristic")
llm_metrics       = QualityMetrics(mode="llm", llm_model="gpt-4o-mini")

print("Setup OK")

In [ ]:
# ── 50 context samples across 5 quality tiers (10 each) ──────────────────────
# Tier 1: Broken/empty  Tier 2: Minimal/generic  Tier 3: Adequate
# Tier 4: Good          Tier 5: Excellent

CONTEXTS = [
    # ── Tier 1: Broken / empty ────────────────────────────────────────────────
    {"tier": 1, "ctx": Context(directive=Directive(content="help"))},
    {"tier": 1, "ctx": Context(directive=Directive(content="do it"))},
    {"tier": 1, "ctx": Context(guidance=Guidance(role=""))},
    {"tier": 1, "ctx": Context(directive=Directive(content="asas"))},
    {"tier": 1, "ctx": Context(guidance=Guidance(role="AI"), directive=Directive(content="answer"))},
    {"tier": 1, "ctx": Context(directive=Directive(content="analyze the following"))},
    {"tier": 1, "ctx": Context(guidance=Guidance(role="Expert Assistant"), directive=Directive(content="be helpful"))},
    {"tier": 1, "ctx": Context(directive=Directive(content="summarize"))},
    {"tier": 1, "ctx": Context(guidance=Guidance(role="helpful assistant"), directive=Directive(content="help me with this"))},
    {"tier": 1, "ctx": Context(directive=Directive(content="do the following"))},

    # ── Tier 2: Minimal / generic ────────────────────────────────────────────
    {"tier": 2, "ctx": Context(
        guidance=Guidance(role="Expert Assistant"),
        directive=Directive(content="Analyze the following problem and provide insights."),
    )},
    {"tier": 2, "ctx": Context(
        guidance=Guidance(role="AI Assistant", rules=["Be helpful", "Be concise"]),
        directive=Directive(content="Summarize the document."),
    )},
    {"tier": 2, "ctx": Context(
        guidance=Guidance(role="Business analyst"),
        directive=Directive(content="Review the situation and recommend solutions."),
    )},
    {"tier": 2, "ctx": Context(
        guidance=Guidance(role="General expert"),
        directive=Directive(content="Explain the concept clearly."),
    )},
    {"tier": 2, "ctx": Context(
        guidance=Guidance(role="Consultant"),
        directive=Directive(content="Identify key issues and provide recommendations."),
    )},
    {"tier": 2, "ctx": Context(
        guidance=Guidance(role="Expert Assistant", rules=["Do your best", "Be clear and helpful"]),
        directive=Directive(content="Help with the task."),
    )},
    {"tier": 2, "ctx": Context(
        guidance=Guidance(role="Senior advisor"),
        directive=Directive(content="Analyze this and suggest improvements."),
    )},
    {"tier": 2, "ctx": Context(
        guidance=Guidance(role="Data analyst"),
        directive=Directive(content="Look at the data and find patterns."),
    )},
    {"tier": 2, "ctx": Context(
        guidance=Guidance(role="Technical expert"),
        directive=Directive(content="Review the code and identify problems."),
    )},
    {"tier": 2, "ctx": Context(
        guidance=Guidance(role="Strategic advisor"),
        directive=Directive(content="Evaluate the situation and provide strategic direction."),
    )},

    # ── Tier 3: Adequate ─────────────────────────────────────────────────────
    {"tier": 3, "ctx": Context(
        guidance=Guidance(
            role="Senior data analyst",
            rules=["Focus on quantitative insights", "Support claims with data"],
        ),
        directive=Directive(content="Analyze customer churn patterns in the Q3 2024 dataset. Identify top 3 risk segments and recommend retention strategies."),
    )},
    {"tier": 3, "ctx": Context(
        guidance=Guidance(role="Software architect", rules=["Consider scalability", "Mention trade-offs"]),
        directive=Directive(content="Review the microservices architecture proposal. Identify bottlenecks and recommend improvements for a system handling 10k req/s."),
    )},
    {"tier": 3, "ctx": Context(
        guidance=Guidance(role="Financial analyst", rules=["Use precise figures", "Focus on material risks"]),
        directive=Directive(content="Assess the financial impact of the proposed acquisition. Focus on revenue synergies, cost structure, and integration risks."),
    )},
    {"tier": 3, "ctx": Context(
        guidance=Guidance(role="Product manager", rules=["Prioritize by user impact", "Be specific about timelines"]),
        directive=Directive(content="Prioritize the Q4 feature roadmap based on customer feedback and revenue potential. Provide a 3-month delivery plan."),
    )},
    {"tier": 3, "ctx": Context(
        guidance=Guidance(role="Security engineer", rules=["Follow OWASP guidelines", "Provide remediation steps"]),
        directive=Directive(content="Audit the authentication system for security vulnerabilities. Classify by severity and provide remediation timeline."),
    )},
    {"tier": 3, "ctx": Context(
        guidance=Guidance(role="HR strategist", rules=["Be objective", "Support with benchmarks"]),
        directive=Directive(content="Design a performance review framework for a 200-person engineering organization. Include rating criteria and calibration process."),
    )},
    {"tier": 3, "ctx": Context(
        guidance=Guidance(role="Market researcher", rules=["Cite data sources", "Distinguish facts from interpretation"]),
        directive=Directive(content="Analyze competitive positioning of the product against top 3 competitors. Focus on pricing, features, and market share."),
    )},
    {"tier": 3, "ctx": Context(
        guidance=Guidance(role="Legal advisor", rules=["Flag material risks", "Reference relevant regulations"]),
        directive=Directive(content="Review the vendor contract for compliance risks under GDPR and SOC 2. Highlight clauses requiring negotiation."),
    )},
    {"tier": 3, "ctx": Context(
        guidance=Guidance(role="Operations manager", rules=["Quantify impact", "Provide actionable steps"]),
        directive=Directive(content="Diagnose the 30% increase in order fulfillment time over the past 6 weeks. Identify root causes and propose immediate fixes."),
    )},
    {"tier": 3, "ctx": Context(
        guidance=Guidance(role="Technical writer", rules=["Use plain language", "Structure clearly"]),
        directive=Directive(content="Write an incident post-mortem for the 4-hour API outage on 2024-11-15. Include timeline, root cause, and prevention measures."),
    )},

    # ── Tier 4: Good ─────────────────────────────────────────────────────────
    {"tier": 4, "ctx": Context(
        guidance=Guidance(
            role="Senior customer success analyst with 8 years of SaaS retention expertise",
            rules=[
                "Apply cohort analysis methodology",
                "Quantify revenue at risk for each segment",
                "Distinguish correlation from causation",
                "Provide 3 prioritized recommendations with 30/60/90-day implementation timeline",
            ],
        ),
        directive=Directive(content="Analyze the 40% churn increase in Q3 2024. Data: cohort breakdown by tier, NPS scores, support ticket themes, and feature usage logs. Output: root causes ranked by confidence level, revenue impact per segment, immediate intervention plan."),
        constraints=Constraints(
            must_include=["cohort analysis", "revenue at risk", "root cause", "NPS correlation"],
            format_rules=["Executive summary first", "Each finding supported by quantified evidence"],
        ),
    )},
    {"tier": 4, "ctx": Context(
        guidance=Guidance(
            role="Principal software engineer specializing in distributed systems and performance optimization",
            rules=[
                "Use the scientific method: measure first, hypothesize, test",
                "Quantify each bottleneck with latency numbers",
                "Consider P50, P95, P99 metrics separately",
                "Recommend load testing methodology",
            ],
        ),
        directive=Directive(content="Diagnose the 3x latency increase in the /api/payments endpoint after v2.4.1 deployment. Instrument list: APM traces, database slow query log, Redis hit/miss ratio, memory profiler. Identify root cause and provide fix with rollback plan."),
        constraints=Constraints(
            must_include=["latency percentiles", "database queries", "root cause", "rollback plan"],
            format_rules=["Start with hypothesis", "Include reproducible test steps"],
        ),
    )},
    {"tier": 4, "ctx": Context(
        guidance=Guidance(
            role="M&A analyst with expertise in SaaS valuation and integration risk",
            rules=[
                "Apply DCF and comparable company analysis",
                "Quantify synergies with assumptions clearly stated",
                "Flag integration risks with probability and impact scores",
                "Apply 3 valuation methodologies and triangulate",
            ],
        ),
        directive=Directive(content="Assess the proposed $45M acquisition of CloudMetrics Inc. Revenue: $8M ARR, 85% gross margin, 120% NRR. Target achieves 40% CAGR. Evaluate strategic fit, valuation multiple, integration complexity, and key risks. Include go/no-go recommendation."),
        constraints=Constraints(
            must_include=["valuation multiple", "integration risk", "synergies", "recommendation"],
            format_rules=["Include sensitivity table", "Executive summary with go/no-go upfront"],
        ),
    )},
    {"tier": 4, "ctx": Context(
        guidance=Guidance(
            role="CISO with deep expertise in cloud security and compliance frameworks",
            rules=[
                "Reference NIST CSF and SOC 2 Type II requirements",
                "Rate each vulnerability by CVSS score",
                "Provide remediation timeline with owner assignments",
                "Distinguish quick wins (< 1 week) from strategic fixes",
            ],
        ),
        directive=Directive(content="Conduct a security architecture review of the production AWS environment. Focus: IAM over-permissioning, unencrypted S3 buckets, missing WAF rules, and exposed RDS endpoints. Output: risk register with CVSS scores, owner assignments, and 30-day remediation roadmap."),
        constraints=Constraints(
            must_include=["CVSS score", "IAM", "remediation timeline", "risk register"],
            format_rules=["Critical findings first", "Each finding: risk/impact/remediation/owner/timeline"],
        ),
    )},
    {"tier": 4, "ctx": Context(
        guidance=Guidance(
            role="OKR coach and strategic planning expert who has implemented OKR systems at 20+ companies",
            rules=[
                "Apply SMART criteria to all objectives",
                "Ensure key results are measurable with leading and lagging indicators",
                "Flag OKRs that are outputs not outcomes",
                "Recommend quarterly check-in cadence",
            ],
        ),
        directive=Directive(content="Review and improve the Q1 2025 OKRs for the platform engineering team (8 engineers, 2 PMs). Current draft: 4 objectives, 12 key results. Identify weak KRs (unmeasurable, output-focused), suggest rewrites, and ensure alignment with company OKRs."),
        constraints=Constraints(
            must_include=["SMART criteria", "leading indicators", "alignment", "rewritten KRs"],
            format_rules=["Show before/after for each KR rewrite", "Alignment matrix at end"],
        ),
    )},
    {"tier": 4, "ctx": Context(
        guidance=Guidance(
            role="Go-to-market strategist with B2B SaaS experience across enterprise and mid-market segments",
            rules=[
                "Base recommendations on customer segmentation data",
                "Quantify each recommendation's revenue potential",
                "Address pricing, positioning, and channel strategy separately",
                "Identify 3 quick wins (< 30 days) and 3 strategic moves (90+ days)",
            ],
        ),
        directive=Directive(content="Develop a GTM strategy for launching the enterprise tier of the analytics platform ($15k/month). TAM: 2,000 mid-to-large companies in financial services. Current ICP: 50-500 seat companies. Competitor pricing analysis, win/loss themes from last 20 deals, and 3 pilot customer profiles included."),
        constraints=Constraints(
            must_include=["ICP", "pricing strategy", "channel strategy", "revenue projection"],
            format_rules=["Include 90-day roadmap", "Segment recommendations by company size"],
        ),
    )},
    {"tier": 4, "ctx": Context(
        guidance=Guidance(
            role="Principal data engineer specializing in real-time data pipelines and event streaming",
            rules=[
                "Compare options on: throughput, latency, cost, operational complexity",
                "Provide benchmarks where available",
                "Include failure scenarios and recovery patterns",
                "Recommend based on the specific requirements given",
            ],
        ),
        directive=Directive(content="Evaluate Kafka vs Kinesis vs Pulsar for real-time event streaming at 500k events/second with < 100ms P99 latency, multi-region replication, and exactly-once semantics. Team: 3 engineers, 6-month delivery deadline. Cloud: AWS. Budget: $20k/month max."),
        constraints=Constraints(
            must_include=["throughput benchmark", "exactly-once semantics", "cost estimate", "recommendation"],
            format_rules=["Comparison matrix first", "Include AWS-specific considerations"],
        ),
    )},
    {"tier": 4, "ctx": Context(
        guidance=Guidance(
            role="Organizational psychologist specializing in performance management and team dynamics",
            rules=[
                "Ground recommendations in peer-reviewed research",
                "Address both individual and team performance dimensions",
                "Provide calibration methodology to reduce bias",
                "Include sample rubrics with behavioral anchors",
            ],
        ),
        directive=Directive(content="Design a bi-annual performance review system for 150 engineers across 4 levels (L3-L6). Requirements: calibrated ratings, bias reduction mechanisms, growth-oriented feedback, and clear promotion criteria. Current pain points: grade inflation, recency bias, manager inconsistency."),
        constraints=Constraints(
            must_include=["calibration process", "behavioral anchors", "bias mitigation", "promotion criteria"],
            format_rules=["Include sample rubric for L4 engineer", "30/60/90 rollout plan"],
        ),
    )},
    {"tier": 4, "ctx": Context(
        guidance=Guidance(
            role="Supply chain strategist with expertise in risk management and resilience planning",
            rules=[
                "Apply SCOR model framework",
                "Quantify risk exposure with probability × impact matrix",
                "Distinguish tactical (< 90 days) from strategic (1+ year) mitigations",
                "Include supplier diversification analysis",
            ],
        ),
        directive=Directive(content="Assess supply chain risk for the APAC manufacturing operations following the Taiwan Strait tensions. Single-source dependencies: 3 critical components from Taiwan, 60% of logistics through Shanghai. Annual revenue at risk: $85M. Output: risk matrix, diversification plan, and 6-month contingency roadmap."),
        constraints=Constraints(
            must_include=["risk matrix", "single-source dependencies", "diversification plan", "contingency roadmap"],
            format_rules=["SCOR model reference", "Risk ranked by revenue impact"],
        ),
    )},
    {"tier": 4, "ctx": Context(
        guidance=Guidance(
            role="Clinical informaticist and healthcare AI specialist with experience in FDA regulatory submissions",
            rules=[
                "Reference FDA AI/ML guidance (2021) and EU AI Act requirements",
                "Distinguish pre-market and post-market surveillance requirements",
                "Address bias and fairness considerations for diverse patient populations",
                "Flag high-risk AI classification triggers",
            ],
        ),
        directive=Directive(content="Evaluate the regulatory compliance requirements for deploying an AI-assisted diagnostic triage system in 12 US hospital networks. Model: deep learning classifier, 94% sensitivity, 89% specificity on internal validation. Use cases: chest X-ray triage, flagging urgent cases for radiologist review."),
        constraints=Constraints(
            must_include=["FDA classification", "bias testing", "post-market surveillance", "clinical validation"],
            format_rules=["Risk classification upfront", "Pre-market vs post-market requirements separated"],
        ),
    )},

    # ── Tier 5: Excellent ─────────────────────────────────────────────────────
    {"tier": 5, "ctx": Context(
        guidance=Guidance(
            role="World-class root cause analyst with expertise in complex systems failure analysis, Six Sigma Black Belt, and experience with NASA FMEA methodology",
            rules=[
                "Apply the Five Whys with each level supported by evidence, not assumptions",
                "Use fault tree analysis to map contributing factors",
                "Distinguish proximate from distal causes",
                "Quantify confidence level (0-100%) for each causal chain",
                "Identify systemic vs one-off failures",
                "Recommend preventive controls with detection lead time",
            ],
        ),
        directive=Directive(content="""Perform a comprehensive root cause analysis of the 4-hour production outage on 2024-11-15 18:00-22:00 UTC affecting the payments API (99.7% error rate, 847k failed transactions, $2.1M revenue impact).

Available data:
- APM traces showing 95th percentile latency: 45ms → 12,000ms at 18:03
- Database slow query log: 847 queries > 30s during window
- Deployment log: v2.4.1 released at 17:58 (5 minutes before outage)
- Redis memory: 94% at outage start, OOM errors at 18:05
- Alert history: memory alert at 17:45 (acknowledged, no action taken)

Output requirements:
1. Executive summary (3 sentences)
2. Timeline of events with contributing factors
3. Five Whys analysis with confidence scores
4. Fault tree diagram (ASCII)
5. Immediate fixes with owner and ETA
6. 30-day prevention roadmap
7. Metrics to detect similar patterns early"""),
        constraints=Constraints(
            must_include=["five whys", "confidence score", "fault tree", "30-day roadmap", "detection metrics"],
            format_rules=[
                "Executive summary must fit in 3 sentences",
                "Each Why must cite specific evidence",
                "Prevention controls must have measurable success criteria",
            ],
        ),
    )},
    {"tier": 5, "ctx": Context(
        guidance=Guidance(
            role="Elite McKinsey-caliber strategy consultant specializing in SaaS business model transformation and enterprise GTM strategy, with 15 years of board-level advisory experience",
            rules=[
                "Ground every claim in quantitative evidence or named research",
                "Apply Minto Pyramid Principle for structure",
                "Distinguish correlation from causation in all data interpretations",
                "Include scenario analysis: base, bull, bear cases with probability weights",
                "Flag assumptions and sensitivity of recommendations to those assumptions",
                "Challenge the brief: identify what is not being asked but should be",
            ],
        ),
        directive=Directive(content="""Develop a comprehensive competitive response strategy for a $200M ARR B2B analytics platform facing a new entrant (YC-backed, $50M raised, 60% lower price point, AI-native architecture).

Context:
- Company: 8-year-old incumbent, 600 enterprise customers, NRR 108%, gross margin 72%
- New entrant: launched 18 months ago, 40 customers, NRR 145%, ML-native data pipeline
- Key threat: 3 enterprise renewals ($12M combined ARR) currently evaluating the competitor
- Internal constraint: 18-month re-architecture timeline to rebuild on modern stack

Deliverables:
1. Competitive positioning analysis (where we win/lose and why)
2. Short-term defense tactics for the 3 at-risk accounts
3. Product strategy: build vs buy vs partner to close capability gap
4. Pricing response: when/how to respond without margin destruction
5. 12-month strategic roadmap with 90-day milestones
6. Board narrative: how to frame this as opportunity, not threat"""),
        constraints=Constraints(
            must_include=["NRR comparison", "pricing strategy", "90-day milestones", "board narrative", "scenario analysis"],
            format_rules=[
                "Minto Pyramid: conclusion first, then evidence",
                "Each recommendation: evidence + assumption + risk if assumption wrong",
                "Scenario table: base/bull/bear with probability and key pivot variables",
            ],
        ),
    )},
    {"tier": 5, "ctx": Context(
        guidance=Guidance(
            role="Principal ML engineer with deep expertise in LLM evaluation, NLP benchmarking, and production ML systems reliability",
            rules=[
                "Apply scientific rigor: define null hypothesis, measurement methodology, and statistical power",
                "Distinguish evaluation at inference time vs training time",
                "Address distribution shift, prompt sensitivity, and model versioning risks",
                "Propose both automated and human evaluation pipelines",
                "Reference established benchmarks (HELM, BIG-Bench, MMLU) where relevant",
                "Include failure mode taxonomy with detection strategies",
            ],
        ),
        directive=Directive(content="""Design a production LLM evaluation framework for a legal document analysis system (contract review, clause extraction, risk flagging) deployed across 200 enterprise law firms.

System specs:
- Model: GPT-4o with RAG over firm-specific document libraries
- Volume: 50,000 documents/month, average 80 pages
- Quality bar: < 0.5% false negative rate on material risk clauses
- Latency: P95 < 30s per document

Required evaluation dimensions:
1. Factual accuracy: extracted clauses vs ground truth
2. Risk classification: precision/recall on 12 risk categories
3. Hallucination rate: claims not grounded in source document
4. Consistency: same document, 3 runs, semantic similarity
5. Latency and throughput at scale

Constraints:
- Limited labeled data: 500 annotated contracts
- 3 annotators with <5% inter-annotator disagreement target
- Continuous evaluation needed (model updates 2x/month)"""),
        constraints=Constraints(
            must_include=["hallucination rate", "inter-annotator agreement", "false negative rate", "continuous evaluation"],
            format_rules=[
                "Include sample annotation guidelines",
                "Define statistical significance thresholds for model update decisions",
                "Separate offline evaluation from online monitoring",
            ],
        ),
    )},
    {"tier": 5, "ctx": Context(
        guidance=Guidance(
            role="Epidemiologist and clinical trial design expert with FDA advisory committee experience and deep knowledge of real-world evidence methodology",
            rules=[
                "Apply ICH E9 statistical analysis principles",
                "Address potential confounders with explicit mitigation strategies",
                "Distinguish efficacy (controlled setting) from effectiveness (real-world)",
                "Calculate required sample size with power analysis assumptions stated",
                "Address regulatory pathway implications for each design choice",
                "Flag ethical considerations per Declaration of Helsinki",
            ],
        ),
        directive=Directive(content="""Design a clinical study protocol to evaluate the effectiveness of an AI-assisted sepsis prediction algorithm in ICU settings.

Algorithm context:
- Input: 47 vitals and lab features, 15-minute prediction window
- Internal validation: AUROC 0.89, sensitivity 82%, specificity 91% (n=12,000 ICU admissions)
- Target: demonstrate 15% reduction in sepsis-related mortality vs standard care

Study constraints:
- Target enrollment: 8 academic medical centers in the US
- Timeline: 18-month enrollment, 6-month follow-up
- Primary endpoint: 28-day all-cause ICU mortality
- Budget: $4.2M
- Regulatory path: FDA De Novo for SaMD Class II device"""),
        constraints=Constraints(
            must_include=["sample size calculation", "primary endpoint", "FDA De Novo", "confounders", "power analysis"],
            format_rules=[
                "CONSORT flow diagram (ASCII)",
                "Statistical analysis plan summary",
                "Risk table: study risks × mitigation × contingency",
            ],
        ),
    )},
    {"tier": 5, "ctx": Context(
        guidance=Guidance(
            role="Chief Architecture Officer with expertise in large-scale distributed systems, cloud-native transformation, and technical due diligence for PE/VC transactions",
            rules=[
                "Apply the C4 model for architecture representation",
                "Quantify technical debt in engineering-weeks",
                "Assess scalability ceiling with load modelling",
                "Evaluate team capability gaps vs architecture requirements",
                "Apply Martin Fowler's refactoring patterns where relevant",
                "Address data sovereignty and compliance architecture",
                "Provide investment-grade risk scoring (1-5 scale with criteria)",
            ],
        ),
        directive=Directive(content="""Conduct a technical due diligence assessment of DataStream Inc. (Series C, $180M valuation) for a potential $45M growth equity investment.

Available artifacts:
- Architecture diagram (monolithic Rails app, PostgreSQL, 3 AWS regions)
- Engineering team: 28 engineers, 4 levels, average tenure 2.1 years
- Codebase stats: 450k LOC, 38% test coverage, 3.2 deployments/day
- Performance: handles 8k req/min peak, P99 latency 340ms
- Incidents: 4 P1 incidents in last 12 months, avg MTTR 4.2 hours
- Roadmap dependency: 3x scale required within 18 months for projected growth

Investment thesis assumption: platform can scale to 25k req/min within 18 months without full rewrite."""),
        constraints=Constraints(
            must_include=["technical debt estimate", "scalability ceiling", "risk score", "team capability assessment", "18-month roadmap"],
            format_rules=[
                "Investment risk score (1-5) with criteria upfront",
                "C4 context diagram (ASCII)",
                "Technical debt breakdown by component in engineering-weeks",
                "Go/no-go recommendation with conditions",
            ],
        ),
    )},
    {"tier": 5, "ctx": Context(
        guidance=Guidance(
            role="Expert regulatory strategist specializing in AI/ML policy, data governance, and cross-jurisdictional compliance for financial services and healthcare sectors",
            rules=[
                "Map requirements to specific regulatory articles/sections",
                "Distinguish must-comply from should-comply requirements",
                "Identify jurisdictional conflicts where regulations contradict",
                "Provide gap analysis against current state with remediation effort estimates",
                "Flag board-level vs operational-level responsibilities",
                "Address algorithmic accountability and explainability requirements",
            ],
        ),
        directive=Directive(content="""Develop a comprehensive regulatory compliance roadmap for a US-based fintech deploying an AI-powered credit scoring model internationally.

Scope: deployment in US, UK, EU, Singapore, and Canada.

Regulatory frameworks to address:
- US: Equal Credit Opportunity Act, Fair Credit Reporting Act, OCC AI guidance
- EU: AI Act (high-risk AI system classification), GDPR Article 22, EBA guidelines
- UK: FCA Consumer Duty, ICO guidance on automated decision-making
- Singapore: MAS FEAT principles, PDPA
- Canada: PIPEDA, proposed AIDA

Model specifics: XGBoost classifier, 180 features, SHAP for explainability, 94% AUC, deployed for loan origination decisions up to $50k."""),
        constraints=Constraints(
            must_include=["EU AI Act", "GDPR Article 22", "SHAP explainability", "gap analysis", "jurisdictional conflicts"],
            format_rules=[
                "Compliance matrix: requirement × jurisdiction × current state × gap × effort",
                "Critical path: items blocking deployment by launch date",
                "Board-level vs operational responsibilities clearly separated",
            ],
        ),
    )},
    {"tier": 5, "ctx": Context(
        guidance=Guidance(
            role="World-class organizational change management expert with experience leading digital transformations at Fortune 500 companies using Kotter's 8-Step and ADKAR frameworks",
            rules=[
                "Apply Kotter's 8-Step Model as the structural backbone",
                "Map stakeholder resistance using Prosci ADKAR",
                "Quantify change saturation risk with the organizational change load model",
                "Include leading indicators for each phase (not just lagging outcomes)",
                "Address culture change separately from process change",
                "Design feedback loops to detect resistance early",
                "Specify communication strategy for each stakeholder archetype",
            ],
        ),
        directive=Directive(content="""Design a 12-month organizational change management plan for implementing AI-assisted decision-making tools across a 3,000-person global insurance underwriting organization.

Context:
- 8 regional offices: US, UK, Germany, Japan, Singapore, Australia, Canada, Brazil
- Current state: manual underwriting, 15-year average employee tenure, high union density in EU offices
- Change scope: replace 60% of manual decisioning with AI recommendations (human-in-the-loop)
- Anticipated resistance: job displacement fear (380 roles affected), regulatory concerns, cultural skepticism
- Success metrics: 85% tool adoption in 12 months, underwriting accuracy +12%, cycle time -40%

Constraints:
- No involuntary layoffs (board commitment)
- EU works council approval required before deployment in EU offices
- Change budget: $2.8M"""),
        constraints=Constraints(
            must_include=["ADKAR", "Kotter", "works council", "change saturation", "leading indicators"],
            format_rules=[
                "Phase-by-phase plan aligned to Kotter's 8 steps",
                "Stakeholder map with resistance level and ADKAR gap",
                "Communication calendar for first 90 days",
                "Leading indicator dashboard definition",
            ],
        ),
    )},
    {"tier": 5, "ctx": Context(
        guidance=Guidance(
            role="Elite competitive intelligence analyst and corporate strategy advisor with deep expertise in technology sector dynamics, patent analysis, and scenario planning",
            rules=[
                "Apply Porter's Five Forces and Jobs-to-be-Done simultaneously",
                "Distinguish signal from noise in competitive intelligence",
                "Use scenario planning (4 quadrants) for market evolution",
                "Quantify competitive moat with specific metrics",
                "Address platform dynamics and network effects explicitly",
                "Flag where data is incomplete and assumptions must be stated",
                "Include second-order effects of strategic moves",
            ],
        ),
        directive=Directive(content="""Develop a 3-year competitive intelligence and strategic positioning report for a vertical AI SaaS company (legal tech, $45M ARR) navigating the emergence of general-purpose LLMs that now perform 70% of their core use cases.

Market context:
- Company: LegalAI Inc., founded 2019, 180 enterprise law firm customers, NPS 62
- Key threat: OpenAI GPT-4o + Copilot being evaluated by 35% of customer base
- Moat: 8M proprietary legal documents (case law, contracts), 12 trained domain models
- Revenue breakdown: 60% usage-based API, 40% enterprise SaaS
- M&A activity: 3 competitors acquired by Big Tech in 18 months

Strategic question: pivot to AI infrastructure layer (sell the data/models) vs double down on application layer vs become acquisition target."""),
        constraints=Constraints(
            must_include=["Porter's Five Forces", "network effects", "scenario planning", "moat quantification", "strategic recommendation"],
            format_rules=[
                "4-scenario quadrant matrix with probability weights",
                "Competitive moat scorecard vs 5 key competitors",
                "Strategic option analysis: 3 paths with NPV estimates",
                "Decision tree for pivot vs stay vs sell",
            ],
        ),
    )},
    {"tier": 5, "ctx": Context(
        guidance=Guidance(
            role="Distinguished Professor of Computer Science and AI ethics, with a research focus on fairness in machine learning, algorithmic accountability, and the societal impacts of AI deployment in high-stakes domains",
            rules=[
                "Apply the IEEE Ethically Aligned Design framework",
                "Distinguish individual fairness from group fairness metrics",
                "Address intersectionality in fairness analysis",
                "Ground recommendations in peer-reviewed ML fairness literature",
                "Consider both technical and sociotechnical interventions",
                "Flag cases where fairness objectives conflict",
                "Address the limits of technical solutions to social problems",
            ],
        ),
        directive=Directive(content="""Conduct a comprehensive algorithmic fairness audit of an AI-powered predictive policing system deployed in 12 US cities for resource allocation (patrol deployment, not predictive arrests).

System specs:
- Input: historical crime reports, 911 calls, socioeconomic census data
- Output: grid-level risk scores (1-10) for 6-hour patrol windows
- Training data: 10 years of crime reports (2012-2022) from 40 cities
- Model: Gradient Boosting with SHAP explanations

Known concerns:
- Audit finding: 23% over-prediction in predominantly Black neighborhoods (vs city average)
- Feedback loop risk: more patrol → more arrests → higher historical crime rate
- First Amendment concerns: chilling effect on lawful assembly in flagged areas"""),
        constraints=Constraints(
            must_include=["fairness metrics", "feedback loop", "demographic parity", "IEEE framework", "sociotechnical interventions"],
            format_rules=[
                "Fairness metric scorecard: 6 metrics with current values and targets",
                "Technical interventions vs policy interventions in separate sections",
                "Stakeholder impact analysis: affected communities, law enforcement, city government",
                "Recommendation: continue/modify/discontinue with conditions",
            ],
        ),
    )},
    {"tier": 5, "ctx": Context(
        guidance=Guidance(
            role="Senior investment banker specializing in cross-border M&A, with expertise in tech sector valuations, deal structuring, and integration planning for transactions above $100M",
            rules=[
                "Apply Sum-of-Parts, DCF, and trading comparables as triangulated valuation",
                "Model synergies bottom-up: cost synergies by function, revenue synergies by product line",
                "Apply integration risk discount to synergy estimates (typically 30-50%)",
                "Address MAC clause triggers and regulatory approval timeline risks",
                "Structure earnout provisions for technology/talent retention",
                "Model accretion/dilution for acquirer across 3 financing structures",
            ],
        ),
        directive=Directive(content="""Develop a comprehensive M&A analysis for CloudSecure Inc. ($95M ARR, 78% gross margin, 115% NRR) as an acquisition target for a strategic buyer (EnterpriseCloud Corp., $2.1B ARR, NASDAQ-listed, $4.5B market cap).

Transaction context:
- CloudSecure specializes in cloud security posture management (CSPM) and CNAPP
- Strategic rationale: fill product gap, expand into CISO buyer, accelerate enterprise motion
- Comparable transactions: Palo Alto / Bridgecrew ($190M, ~18x ARR), Wiz growth multiple
- CloudSecure profile: 280 customers, 95% retention, 60 engineers, 3 patent filings
- Current discussions: indicative range $450-520M (4.7-5.5x ARR)

Required analysis:
1. Valuation range with bull/base/bear scenarios
2. Synergy model (cost + revenue) with integration timeline
3. Deal structure recommendation (cash vs stock vs earnout mix)
4. Key risks and MAC clause considerations
5. Integration 100-day plan focusing on product and go-to-market"""),
        constraints=Constraints(
            must_include=["DCF valuation", "synergy model", "accretion/dilution", "deal structure", "integration 100-day plan"],
            format_rules=[
                "Valuation bridge chart (ASCII)",
                "Synergy model by function with integration timeline",
                "Accretion/dilution table for 3 financing structures",
                "Risk register with MAC clause triggers",
            ],
        ),
    )},
]

print(f"Total contexts: {len(CONTEXTS)}")
print(f"Per tier: {pd.Series([c['tier'] for c in CONTEXTS]).value_counts().sort_index().to_dict()}")

In [ ]:
# ── Phase 1: Score all contexts with both heuristic and LLM mode ─────────────
# WARNING: LLM scoring costs ~$0.02/eval × 50 contexts ≈ $1.00 total
# Cached to calibration_scores.csv — rerun only if FORCE_RERUN = True

CACHE_FILE = pathlib.Path("calibration_scores.csv")
FORCE_RERUN = False   # set to True to re-score everything

if CACHE_FILE.exists() and not FORCE_RERUN:
    scores_df = pd.read_csv(CACHE_FILE)
    print(f"Loaded from cache: {len(scores_df)} rows")
else:
    print("Scoring all contexts (heuristic + LLM)...")
    rows = []
    dims = list(QualityDimension)

    for i, item in enumerate(CONTEXTS):
        ctx = item["ctx"]
        tier = item["tier"]
        try:
            h_score = heuristic_metrics.evaluate(ctx)
            l_score = llm_metrics.evaluate(ctx)

            row = {"context_idx": i, "tier": tier,
                   "heuristic_overall": h_score.overall,
                   "llm_overall": l_score.overall}
            for dim in dims:
                row[f"heuristic_{dim.value}"] = h_score.dimensions.get(dim, 0)
                row[f"llm_{dim.value}"] = l_score.dimensions.get(dim, 0)
            rows.append(row)
            print(f"  [{i+1:2d}/50] tier={tier} heuristic={h_score.overall:.2f} llm={l_score.overall:.2f}")
        except Exception as e:
            print(f"  [{i+1:2d}/50] ERROR: {e}")

    scores_df = pd.DataFrame(rows)
    scores_df.to_csv(CACHE_FILE, index=False)
    print(f"\nSaved to {CACHE_FILE}")

print(scores_df[["tier","heuristic_overall","llm_overall"]].groupby("tier").mean().round(3))

In [ ]:
# ── Spearman correlations: heuristic vs LLM ───────────────────────────────────

dims = list(QualityDimension)

print("Spearman ρ (heuristic vs LLM):")
print(f"  {'Dimension':<20} {'ρ':>8} {'p-value':>10} {'Assessment'}")
print("  " + "-" * 55)

corr_rows = []
for dim in dims:
    h_col = f"heuristic_{dim.value}"
    l_col = f"llm_{dim.value}"
    if h_col in scores_df.columns and l_col in scores_df.columns:
        rho, pval = stats.spearmanr(scores_df[h_col], scores_df[l_col])
        status = "✅ validated" if rho > 0.65 else ("⚠️ marginal" if rho > 0.40 else "❌ needs work")
        print(f"  {dim.value:<20} {rho:>8.3f} {pval:>10.4f}  {status}")
        corr_rows.append({"dimension": dim.value, "rho": rho, "pval": pval})

rho_overall, pval_overall = stats.spearmanr(scores_df["heuristic_overall"], scores_df["llm_overall"])
status_overall = "✅ validated" if rho_overall > 0.65 else ("⚠️ marginal" if rho_overall > 0.40 else "❌ needs work")
print(f"\n  {'OVERALL':<20} {rho_overall:>8.3f} {pval_overall:>10.4f}  {status_overall}")

corr_df = pd.DataFrame(corr_rows)

In [ ]:
# ── Visualizations ───────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

tier_colors = {1: "#e74c3c", 2: "#e67e22", 3: "#f1c40f", 4: "#2ecc71", 5: "#3498db"}
colors = [tier_colors[t] for t in scores_df["tier"]]

# Plot 1: overall heuristic vs LLM
ax = axes[0]
ax.scatter(scores_df["heuristic_overall"], scores_df["llm_overall"], c=colors, alpha=0.8, s=60)
ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Perfect agreement")
ax.set_xlabel("Heuristic Score")
ax.set_ylabel("LLM Score")
ax.set_title(f"Overall: ρ={rho_overall:.3f}")
ax.legend(fontsize=8)
for tier, color in tier_colors.items():
    ax.scatter([], [], c=color, label=f"Tier {tier}")
ax.legend(fontsize=7, ncol=2)

# Plots 2-6: per dimension
for i, dim in enumerate(dims[:5]):
    ax = axes[i + 1]
    h_col = f"heuristic_{dim.value}"
    l_col = f"llm_{dim.value}"
    rho = corr_df[corr_df["dimension"] == dim.value]["rho"].values[0]
    ax.scatter(scores_df[h_col], scores_df[l_col], c=colors, alpha=0.8, s=50)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax.set_xlabel("Heuristic")
    ax.set_ylabel("LLM")
    ax.set_title(f"{dim.value.title()}: ρ={rho:.3f}")

plt.suptitle("QualityMetrics Heuristic vs LLM — Phase 1 Calibration Study", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../calibration_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: research/calibration_scatter.png")

In [ ]:
# ── Tier separation analysis ─────────────────────────────────────────────────
# Does the heuristic correctly rank quality tiers?

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col, label in [
    (axes[0], "heuristic_overall", "Heuristic"),
    (axes[1], "llm_overall", "LLM (GPT-4o-mini)"),
]:
    tier_means = scores_df.groupby("tier")[col].agg(["mean", "std"]).reset_index()
    ax.bar(tier_means["tier"], tier_means["mean"],
           yerr=tier_means["std"], capsize=5,
           color=[tier_colors[t] for t in tier_means["tier"]], alpha=0.85)
    ax.set_xlabel("Quality Tier")
    ax.set_ylabel("Score")
    ax.set_title(f"{label} Scores by Tier")
    ax.set_ylim(0, 1.1)
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_xticklabels(["1\nBroken", "2\nMinimal", "3\nAdequate", "4\nGood", "5\nExcellent"])

    # Check monotonicity
    means = tier_means["mean"].values
    monotone = all(means[i] <= means[i+1] for i in range(len(means)-1))
    ax.set_xlabel(f"Quality Tier {'✅ monotone' if monotone else '❌ not monotone'}")

plt.suptitle("Score vs Quality Tier: Does the scorer rank correctly?", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("../calibration_tiers.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: research/calibration_tiers.png")

# Kendall's tau for ordinal rank agreement
from scipy.stats import kendalltau
tau, p_tau = kendalltau(scores_df["tier"], scores_df["heuristic_overall"])
print(f"\nKendall's τ (tier vs heuristic): {tau:.3f} (p={p_tau:.4f})")
print("Interpretation: τ > 0.6 = strong rank agreement with quality tier")

In [ ]:
# ── Phase 2: Human annotation (run when annotators available) ─────────────────

HUMAN_RATINGS_FILE = pathlib.Path("human_ratings.csv")

# Export contexts for annotation
if not HUMAN_RATINGS_FILE.exists():
    export_rows = []
    for i, item in enumerate(CONTEXTS):
        assembled = item["ctx"].assemble()[:500]  # truncate for readability
        export_rows.append({
            "context_idx": i,
            "tier_ground_truth": item["tier"],
            "assembled_preview": assembled,
            # Annotator fills these columns:
            "annotator_1_overall": "",
            "annotator_1_clarity": "",
            "annotator_1_completeness": "",
            "annotator_1_specificity": "",
            "annotator_1_relevance": "",
            "annotator_1_structure": "",
            "annotator_1_efficiency": "",
            "annotator_2_overall": "",
            "annotator_2_clarity": "",
            "annotator_2_completeness": "",
            "annotator_2_specificity": "",
            "annotator_2_relevance": "",
            "annotator_2_structure": "",
            "annotator_2_efficiency": "",
            "annotator_3_overall": "",
            "annotator_3_clarity": "",
            "annotator_3_completeness": "",
            "annotator_3_specificity": "",
            "annotator_3_relevance": "",
            "annotator_3_structure": "",
            "annotator_3_efficiency": "",
        })
    pd.DataFrame(export_rows).to_csv(HUMAN_RATINGS_FILE, index=False)
    print(f"Created annotation template: {HUMAN_RATINGS_FILE}")
    print("Instructions: Fill in columns annotator_1/2/3_* with scores 0.0-1.0")
    print("Then re-run this cell to load and analyze.")
else:
    # Load completed ratings
    hr = pd.read_csv(HUMAN_RATINGS_FILE)
    hr = hr[hr["annotator_1_overall"] != ""]  # filter rows with ratings

    if len(hr) == 0:
        print("Annotation file exists but no ratings filled in yet.")
    else:
        print(f"Loaded {len(hr)} annotated contexts")
        annotators = [1, 2, 3]
        dims = ["overall", "clarity", "completeness", "specificity", "relevance", "structure", "efficiency"]

        # Inter-annotator agreement (simplified Krippendorff's alpha approx via ICC)
        from scipy.stats import pearsonr
        print("\nInter-annotator agreement (Pearson r for overall scores):")
        for a1, a2 in [(1,2),(1,3),(2,3)]:
            r, p = pearsonr(hr[f"annotator_{a1}_overall"].astype(float),
                           hr[f"annotator_{a2}_overall"].astype(float))
            print(f"  Annotator {a1} vs {a2}: r={r:.3f} (p={p:.4f})")

        # Average human scores
        hr["human_mean_overall"] = hr[[f"annotator_{a}_overall" for a in annotators]].astype(float).mean(axis=1)

        # Correlate heuristic vs human
        merged = hr.merge(scores_df[["context_idx", "heuristic_overall"]], on="context_idx")
        rho_human, p_human = stats.spearmanr(merged["heuristic_overall"], merged["human_mean_overall"])
        print(f"\nHeuristic vs Human (Spearman ρ): {rho_human:.3f} (p={p_human:.4f})")
        status = "✅ validated" if rho_human > 0.65 else "⚠️ needs improvement"
        print(f"Status: {status}")

## Results Interpretation

### Phase 1 (LLM proxy)
- **Overall ρ > 0.65**: heuristic is well-calibrated with LLM judge → deploy with confidence
- **Overall ρ 0.40–0.65**: reasonable but room for improvement → investigate failing dimensions
- **Overall ρ < 0.40**: significant calibration gap → needs rework before relying on heuristic scores
- **Kendall's τ > 0.6**: scorer reliably separates quality tiers in rank order

### Per-dimension guidance
| ρ range | Action |
|---------|--------|
| > 0.75 | Strong signal — no changes needed |
| 0.50–0.75 | Adequate — monitor for edge cases |
| 0.30–0.50 | Weak — investigate top failures, add missing heuristics |
| < 0.30 | Broken — overhaul needed |

### Phase 2 (Human)
- Target inter-annotator agreement: Pearson r > 0.70 between annotators
- Target heuristic-human ρ: > 0.60

### Weight recalibration
After Phase 2, if a dimension shows consistently high human importance but low weight in current scoring, update `self.weights` in `QualityMetrics.__init__`.